# DichVideo Batch OmniVoice from SRT

Upload many `.srt` files and one reference voice audio. This notebook uses a high-quality OmniVoice profile: torch CUDA 12.8, OmniVoice from GitHub, faster-whisper ref transcription, a 20 second mono 24 kHz reference clip, 32 inference steps, and 24-bit WAV output.

Use only with your own voice or with clear permission from the voice owner. Runtime > Change runtime type > GPU before running.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README recommends torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile


In [ ]:
# Use models already uploaded to Google Drive. This cell does not download model weights.
# Expected Drive folders:
# - MyDrive/models/OmniVoice
# - MyDrive/models/faster-whisper-large-v3
USE_DRIVE_MODELS = True
OMNIVOICE_MODEL_FOLDER = 'OmniVoice'
ASR_MODEL_FOLDER = 'faster-whisper-large-v3'

if USE_DRIVE_MODELS:
    from google.colab import drive
    from pathlib import Path
    import shutil

    drive.mount('/content/drive')
    drive_models_root = Path('/content/drive/MyDrive/models')
    local_models_root = Path('/content/models')

    def copy_drive_model(folder_name: str, required_file: str | None = None) -> str:
        drive_dir = drive_models_root / folder_name
        local_dir = local_models_root / folder_name
        if required_file and not (drive_dir / required_file).exists():
            raise FileNotFoundError(
                'Model not found in Google Drive. Upload the full model folder so this file exists: '
                f'{drive_dir / required_file}'
            )
        if not required_file and (not drive_dir.exists() or not any(drive_dir.iterdir())):
            raise FileNotFoundError(
                'Model folder not found in Google Drive. Upload the full model folder here: '
                f'{drive_dir}'
            )
        if not local_dir.exists():
            print(f'Copying {drive_dir} -> {local_dir}')
            local_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(drive_dir, local_dir)
        else:
            print(f'Using local model copy: {local_dir}')
        return str(local_dir)

    OMNIVOICE_MODEL_PATH = copy_drive_model(OMNIVOICE_MODEL_FOLDER)
    ASR_MODEL_PATH = copy_drive_model(ASR_MODEL_FOLDER, required_file='model.bin')
else:
    OMNIVOICE_MODEL_PATH = 'k2-fsa/OmniVoice'
    ASR_MODEL_PATH = 'medium'

print('OMNIVOICE_MODEL_PATH =', OMNIVOICE_MODEL_PATH)
print('ASR_MODEL_PATH =', ASR_MODEL_PATH)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

SRT_DIR = Path('/content/dichvideo_srt_uploads')
OUTPUT_DIR = Path('/content/dichvideo_omnivoice_audio')
shutil.rmtree(SRT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
SRT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix == '.srt':
        (SRT_DIR / name).write_bytes(data)
    elif suffix in {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')

print('Reference audio:', REF_AUDIO)
print('SRT files:')
for path in sorted(SRT_DIR.glob('*.srt')):
    print('-', path.name)


In [ ]:
import html
import re
import subprocess
import torch
import unicodedata
from faster_whisper import WhisperModel

REFERENCE_WAV = '/content/ref.wav'
REFERENCE_START = '0'
REFERENCE_DURATION = '8'
LANGUAGE_ID = 'vi'
ASR_MODEL = ASR_MODEL_PATH

# Best-quality default: use 3-10 seconds of clean same-language speech, mono 24 kHz.
# If the source has leading silence, change REFERENCE_START to where speech begins.
subprocess.run([
    'ffmpeg', '-y', '-i', REF_AUDIO,
    '-ss', REFERENCE_START, '-t', REFERENCE_DURATION,
    '-vn', '-af', 'atrim=start=0,asetpts=PTS-STARTPTS,loudnorm=I=-18:TP=-2:LRA=11',
    '-ar', '24000', '-ac', '1',
    REFERENCE_WAV,
], check=True)

print('Reference WAV:', REFERENCE_WAV)

def clean_reference_text(text: str) -> str:
    text = html.unescape(unicodedata.normalize('NFKC', text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\{[^{}]*\}', ' ', text)
    text = re.sub(r'["“”‘’`]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text and text[-1] not in '.,!?;:':
        text += '.'
    return text

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
asr = WhisperModel(ASR_MODEL, device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language=LANGUAGE_ID, vad_filter=True)
REF_TEXT = clean_reference_text(' '.join(seg.text.strip() for seg in segments).strip())
print('REF_TEXT =', REF_TEXT)
if not REF_TEXT:
    raise RuntimeError('Whisper did not recognize text from the reference audio. Use a clearer reference clip.')


In [ ]:
# Embedded worker: creates normalized, padded segments/*.wav for safer OmniVoice batch TTS.
from pathlib import Path

WORKER_PATH = '/content/colab_batch_omnivoice_from_srt.py'
Path(WORKER_PATH).write_text('from __future__ import annotations\n\nimport argparse\nimport html\nimport json\nimport logging\nimport re\nimport shutil\nimport subprocess\nimport time\nimport unicodedata\nfrom pathlib import Path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Batch OmniVoice TTS from uploaded SRT files.")\n    parser.add_argument("--srt-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--ref-audio", required=True)\n    parser.add_argument("--ref-text", default=None)\n    parser.add_argument("--model", default="k2-fsa/OmniVoice")\n    parser.add_argument("--device", default="cuda:0")\n    parser.add_argument("--dtype", default="float16", choices=["float16", "float32"])\n    parser.add_argument("--reference-start", type=float, default=0.0)\n    parser.add_argument("--reference-duration", type=float, default=8.0)\n    parser.add_argument("--speed", type=float, default=0.95)\n    parser.add_argument("--trim-start-seconds", type=float, default=0.0)\n    parser.add_argument("--end-padding-seconds", type=float, default=0.35)\n    parser.add_argument("--num-step", type=int, default=32)\n    args = parser.parse_args()\n\n    srt_dir = Path(args.srt_dir)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    logger = _setup_logger(output_dir / "batch_omnivoice_from_srt.log")\n    _check_binary("ffmpeg")\n    _check_binary("ffprobe")\n\n    srt_files = sorted(srt_dir.glob("*.srt"))\n    if not srt_files:\n        raise RuntimeError(f"No .srt files found in {srt_dir}")\n\n    ref_audio = _prepare_reference_audio(\n        Path(args.ref_audio),\n        output_dir / "reference_24k.wav",\n        args.reference_start,\n        args.reference_duration,\n        logger,\n    )\n\n    import soundfile as sf\n    import torch\n    from omnivoice import OmniVoice\n\n    dtype = torch.float16 if args.dtype == "float16" else torch.float32\n    logger.info("Loading OmniVoice model=%s device=%s dtype=%s", args.model, args.device, args.dtype)\n    model = OmniVoice.from_pretrained(args.model, device_map=args.device, dtype=dtype)\n\n    for srt_index, srt_path in enumerate(srt_files, start=1):\n        logger.info("Processing SRT %s/%s: %s", srt_index, len(srt_files), srt_path)\n        segments = parse_srt(srt_path.read_text(encoding="utf-8-sig"))\n        if not segments:\n            logger.warning("Skipping empty SRT: %s", srt_path)\n            continue\n        _process_one_srt(\n            model=model,\n            sf=sf,\n            srt_path=srt_path,\n            segments=segments,\n            ref_audio=ref_audio,\n            ref_text=args.ref_text,\n            output_dir=output_dir,\n            speed=args.speed,\n            trim_start_seconds=args.trim_start_seconds,\n            end_padding_seconds=args.end_padding_seconds,\n            num_step=args.num_step,\n            logger=logger,\n        )\n\n    zip_base = output_dir.parent / "omnivoice_audio_results"\n    if zip_base.with_suffix(".zip").exists():\n        zip_base.with_suffix(".zip").unlink()\n    shutil.make_archive(str(zip_base), "zip", output_dir)\n    logger.info("Created zip: %s.zip", zip_base)\n\n\ndef _process_one_srt(model, sf, srt_path: Path, segments: list[dict], ref_audio: Path, ref_text: str | None, output_dir: Path, speed: float, trim_start_seconds: float, end_padding_seconds: float, num_step: int, logger: logging.Logger) -> None:\n    name = _safe_stem(srt_path)\n    per_srt_dir = output_dir / name\n    raw_dir = per_srt_dir / "segments"\n    raw_dir.mkdir(parents=True, exist_ok=True)\n\n    scheduled = []\n    cursor = 0.0\n    for seg in segments:\n        original_text = " ".join(seg["text"].split())\n        text = _prepare_tts_text(original_text)\n        raw_wav = raw_dir / f"{seg[\'index\']:04d}.wav"\n        logger.info(\n            "OmniVoice generate srt=%s segment=%s chars=%s normalized_chars=%s",\n            srt_path.name,\n            seg["index"],\n            len(original_text),\n            len(text),\n        )\n        audio = model.generate(\n            text=text,\n            ref_audio=str(ref_audio),\n            ref_text=ref_text,\n            speed=speed,\n            num_step=num_step,\n        )\n        audio_data = _trim_start(audio[0], sample_rate=24000, trim_seconds=trim_start_seconds)\n        sf.write(str(raw_wav), audio_data, 24000, subtype="PCM_24")\n        if end_padding_seconds > 0:\n            _pad_audio_tail(raw_wav, end_padding_seconds, logger)\n\n        raw_duration = _duration(raw_wav, logger)\n        scheduled_duration = raw_duration\n        scheduled_start = max(seg["start"], cursor)\n        cursor = scheduled_start + scheduled_duration\n\n        scheduled.append({\n            **seg,\n            "tts_text": text,\n            "scheduled_start": scheduled_start,\n            "scheduled_end": scheduled_start + scheduled_duration,\n            "audio_duration": scheduled_duration,\n            "audio": str(raw_wav.relative_to(per_srt_dir)),\n        })\n\n    full_wav = per_srt_dir / f"{name}_full.wav"\n    _mix_scheduled_audio(scheduled, full_wav, logger, per_srt_dir)\n    (per_srt_dir / f"{name}.srt").write_text(srt_path.read_text(encoding="utf-8-sig"), encoding="utf-8")\n    _write_json(per_srt_dir / "timing_schedule.json", scheduled)\n    logger.info("Finished %s -> %s", srt_path.name, full_wav)\n\n\ndef parse_srt(content: str) -> list[dict]:\n    content = content.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n    if not content:\n        return []\n    blocks = re.split(r"\\n\\s*\\n", content)\n    segments = []\n    fallback_index = 1\n    for block in blocks:\n        lines = [line.strip() for line in block.split("\\n") if line.strip()]\n        if not lines:\n            continue\n        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)\n        if timing_line_index is None:\n            continue\n        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)\n        try:\n            index = int(re.sub(r"\\D+", "", maybe_index) or fallback_index)\n        except ValueError:\n            index = fallback_index\n        timing = lines[timing_line_index]\n        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]\n        text = " ".join(lines[timing_line_index + 1:]).strip()\n        if text:\n            segments.append({\n                "index": index,\n                "start": _parse_srt_timestamp(start_s),\n                "end": _parse_srt_timestamp(end_s),\n                "text": text,\n            })\n            fallback_index += 1\n    return segments\n\n\nCASE_SENSITIVE_READINGS = {\n    "AI": "ây ai",\n    "API": "ây pi ai",\n    "CPU": "si pi diu",\n    "GPU": "gi pi diu",\n    "CEO": "si i ô",\n    "USB": "diu ét bi",\n    "USD": "đô la Mỹ",\n    "USA": "Mỹ",\n    "UK": "Anh",\n    "OK": "ô kê",\n}\n\n\nLETTER_READINGS = {\n    "A": "ây", "B": "bi", "C": "si", "D": "đi", "E": "i", "F": "ép",\n    "G": "gi", "H": "hát", "I": "ai", "J": "giây", "K": "kây", "L": "eo",\n    "M": "em", "N": "en", "O": "ô", "P": "pi", "Q": "kiu", "R": "a",\n    "S": "ét", "T": "ti", "U": "diu", "V": "vi", "W": "đắp liu", "X": "ích",\n    "Y": "goai", "Z": "di",\n}\n\n\ndef _apply_pronunciation_dictionary(text: str) -> str:\n    for source, target in sorted(CASE_SENSITIVE_READINGS.items(), key=lambda item: len(item[0]), reverse=True):\n        text = re.sub(r"(?<![\\w])" + re.escape(source) + r"(?![\\w])", target, text)\n    return re.sub(\n        r"(?<![\\w])([A-Z]{2,})(?![\\w])",\n        lambda match: " ".join(LETTER_READINGS.get(char, char) for char in match.group(1)),\n        text,\n    )\n\n\ndef _prepare_tts_text(text: str) -> str:\n    text = html.unescape(unicodedata.normalize("NFKC", text))\n    text = re.sub(r"<[^>]+>", " ", text)\n    text = re.sub(r"\\{[^{}]*\\}", " ", text)\n    text = text.replace("…", ", ").replace("...", ", ")\n    replacements = {\n        "%": " phần trăm ",\n        "&": " và ",\n        "@": " a còng ",\n        "#": " số ",\n        "$": " đô la ",\n        "+": " cộng ",\n        "=": " bằng ",\n        "*": " sao ",\n        "×": " nhân ",\n        "÷": " chia ",\n        "/": " trên ",\n        "\\\\": " ",\n        "|": " ",\n        "_": " ",\n        "~": " ",\n        "^": " ",\n    }\n    for source, target in replacements.items():\n        text = text.replace(source, target)\n    text = _apply_pronunciation_dictionary(text)\n    text = re.sub(r"[\\[\\]{}()<>]", ", ", text)\n    text = re.sub(r"[\\"\'“”‘’`]+", "", text)\n    text = re.sub(r"\\s*[-–—]+\\s*", ", ", text)\n    text = re.sub(r"\\s+", " ", text).strip()\n    if text and text[-1] not in ".,!?;:":\n        text += "."\n    return text\n\n\ndef _parse_srt_timestamp(value: str) -> float:\n    match = re.match(r"(\\d+):(\\d+):(\\d+)[,.](\\d+)", value)\n    if not match:\n        raise ValueError(f"Invalid SRT timestamp: {value}")\n    h, m, s, ms = match.groups()\n    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000\n\n\ndef _mix_scheduled_audio(scheduled: list[dict], output_path: Path, logger: logging.Logger, root_dir: Path) -> None:\n    total_duration = max(float(item["scheduled_end"]) for item in scheduled)\n    silence_path = root_dir / "_silence.wav"\n    _run(["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=channel_layout=mono:sample_rate=44100", "-t", f"{total_duration:.3f}", str(silence_path)], logger)\n    inputs = ["-i", str(silence_path)]\n    filters = []\n    mix_inputs = ["[0:a]"]\n    for input_index, item in enumerate(scheduled, start=1):\n        audio_path = root_dir / item["audio"]\n        inputs.extend(["-i", str(audio_path)])\n        delay_ms = max(0, int(float(item["scheduled_start"]) * 1000))\n        label = f"a{input_index}"\n        filters.append(f"[{input_index}:a]adelay={delay_ms}:all=1[{label}]")\n        mix_inputs.append(f"[{label}]")\n    filter_complex = ";".join(filters + [f"{\'\'.join(mix_inputs)}amix=inputs={len(mix_inputs)}:normalize=0[out]"])\n    _run(["ffmpeg", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[out]", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)\n    silence_path.unlink(missing_ok=True)\n\n\ndef _prepare_reference_audio(input_path: Path, output_path: Path, start: float, duration: float, logger: logging.Logger) -> Path:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    _run([\n        "ffmpeg", "-y", "-i", str(input_path),\n        "-ss", f"{start:.3f}", "-t", f"{duration:.3f}",\n        "-vn", "-af", "atrim=start=0,asetpts=PTS-STARTPTS,loudnorm=I=-18:TP=-2:LRA=11",\n        "-ac", "1", "-ar", "24000", "-c:a", "pcm_s24le",\n        str(output_path),\n    ], logger)\n    return output_path\n\n\ndef _trim_start(audio_data, sample_rate: int, trim_seconds: float):\n    trim_samples = max(0, int(sample_rate * trim_seconds))\n    if trim_samples <= 0:\n        return audio_data\n    if len(audio_data) <= trim_samples:\n        return audio_data\n    return audio_data[trim_samples:]\n\n\ndef _pad_audio_tail(path: Path, pad_seconds: float, logger: logging.Logger) -> None:\n    temp_path = path.with_name(path.stem + "_pad" + path.suffix)\n    _run([\n        "ffmpeg", "-y", "-i", str(path),\n        "-filter:a", f"apad=pad_dur={pad_seconds:.3f}",\n        "-ac", "1", "-ar", "24000", "-c:a", "pcm_s24le",\n        str(temp_path),\n    ], logger)\n    temp_path.replace(path)\n\n\ndef _duration(path: Path, logger: logging.Logger) -> float:\n    completed = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)], logger)\n    return float(completed.stdout.strip())\n\n\ndef _safe_stem(path: Path) -> str:\n    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", path.stem).strip("._")\n    return stem or "srt"\n\n\ndef _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:\n    logger.info("Running command: %s", " ".join(cmd))\n    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if completed.stdout.strip():\n        logger.info("stdout: %s", completed.stdout.strip()[-3000:])\n    if completed.stderr.strip():\n        logger.info("stderr: %s", completed.stderr.strip()[-3000:])\n    if completed.returncode != 0:\n        raise RuntimeError(f"Command failed with code {completed.returncode}: {\' \'.join(cmd)}")\n    return completed\n\n\ndef _check_binary(name: str) -> None:\n    if shutil.which(name) is None:\n        raise RuntimeError(f"Missing dependency: {name}")\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef _setup_logger(log_path: Path) -> logging.Logger:\n    logger = logging.getLogger("dichvideo_batch_omnivoice_from_srt")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n    return logger\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('WORKER_PATH:', WORKER_PATH)


In [ ]:
import torch

MODEL = OMNIVOICE_MODEL_PATH
NUM_STEP = '64'  # Higher quality/stability for final output; use 32 only when you need faster output.
SPEED = '1.00'  # Slightly slower helps OmniVoice pronounce Vietnamese and punctuation-heavy text more reliably.
TRIM_START_SECONDS = '0.0'  # Avoid cutting the first phoneme; raise only if your model adds a fixed click/silence.
END_PADDING_SECONDS = '0.35'  # Prevent the final word of each segment from being clipped during mux/sync.
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'

!python "$WORKER_PATH" \
  --srt-dir "$SRT_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --ref-audio "$REFERENCE_WAV" \
  --ref-text "$REF_TEXT" \
  --reference-start "0" \
  --reference-duration "9999" \
  --model "$MODEL" \
  --device "$DEVICE" \
  --dtype "$DTYPE" \
  --speed "$SPEED" \
  --trim-start-seconds "$TRIM_START_SECONDS" \
  --end-padding-seconds "$END_PADDING_SECONDS" \
  --num-step "$NUM_STEP"


In [ ]:
from google.colab import files
files.download('/content/omnivoice_audio_results.zip')
